# 07 · Apply the manuscript’s exact E-band selection and plot WVD hexbin

**Depends on 06.** Uses the original paper-support plotting code and the *actual* pairs/population metrics created by `33`. Selection uses **path length ≥4 km**, **quantization-limited WVD resolution ≤0.20 g m⁻³**, **71–76 or 81–86 GHz**, **GRAF branch**. It produces manuscript-style hexbin(s) and `wvp_support_metrics.csv`. Sample counts must be checked against the paper (31 physical links/45 directional sublinks in the previous validation); do not force counts.

In [1]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)


Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow


In [2]:
from essd_cz.connected import plot_paper_wvd
project.get('wvd_binned_metrics')   # ensures the last notebook actually ran
PLOT_MAX_WVD=35.0
GRID_SIZE=55
DPI=300
out=plot_paper_wvd(project,dpi=DPI,gridsize=GRID_SIZE,plot_max=PLOT_MAX_WVD)
print('Hexbins/metrics:',out)
import pandas as pd
print(pd.read_csv(project.get('wvd_paper_metrics')).to_string(index=False))

ORIGINAL WVD PAPER HEXBIN: /home/jovyan/.venvs/pr13_xarray_2026_04/bin/python /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/original/code/wvd/plot_wvp_support_hexbins_no_titles.py --root /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow --out-dir /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/07_wvd_paper_population --dpi 300 --gridsize 55 --plot-max 35.0
Selection:
  length >= 4.0 km
  Delta rho_q <= 0.20 g m-3

Metrics:
population  min_length_km  max_delta_rho_q_gm3  N_observations  n_sublinks  n_links      PCC  RMSE_gm3  Bias_gm3
All E-band            4.0                  0.2          236793          45       31 0.598345  3.459552  0.012654
 81–86 GHz            4.0                  0.2          161307          31       31 0.554342  3.689960  0.036313
 71–76 GHz            4.0                  0.2           75486          14       14 0.700613  2.

In [3]:
from essd_cz.connected import Project
import pandas as pd

# Reload the latest outputs from Notebook 06
project = Project(CONFIG, WORKDIR)

folder = project.output / "WVD_V6_QUANTIZATION_BINNED_VALIDATION"

required = {
    "Sensitivity metrics": folder / "binned_validation_metrics.csv",
    "Population": folder / "validation_population_links.csv",
    "GRAF validation pairs": folder / "validation_pairs_graf.npz",
}

for name, path in required.items():
    print(f"{name}: {'OK' if path.is_file() else 'MISSING'}")
    print(" ", path)

print("Registered stages:", list(project.state["stages"]))

if all(path.is_file() for path in required.values()):
    links = pd.read_csv(required["Population"])

    selected = links.loc[
        (
            links["frequency_ghz"].between(71, 76)
            | links["frequency_ghz"].between(81, 86)
        )
        & (links["length_km"] >= 4)
        & (
            links["wvd_quantization_resolution_ref20c_rho10_gm3"]
            <= 0.20
        )
    ]

    print("Selected directional sublinks:", len(selected))
    print("Selected physical CMLs:", selected["cml_id"].nunique())

Sensitivity metrics: OK
  /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/WVD_V6_QUANTIZATION_BINNED_VALIDATION/binned_validation_metrics.csv
Population: OK
  /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/WVD_V6_QUANTIZATION_BINNED_VALIDATION/validation_population_links.csv
GRAF validation pairs: OK
  /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/WVD_V6_QUANTIZATION_BINNED_VALIDATION/validation_pairs_graf.npz
Registered stages: ['exploration_complete', 'rainfall', 'rain_timeline', 'spatial_grid', 'wvd_v6', 'map_hourly', 'wvd_binned_metrics', 'wvd_link_population', 'wvd_paper_metrics']
Selected directional sublinks: 45
Selected physical CMLs: 31
